# Replikasi Klasifikasi Penyakit Daun Tomat: SVM vs KNN

Notebook ini menjalankan project secara step-by-step agar setiap tahap eksperimen dapat dibaca, dieksekusi ulang, dan diperiksa hasilnya.

Metode yang digunakan:

- Support Vector Machine (SVM)
- K-Nearest Neighbor (KNN)

Fitur citra yang diekstraksi:

- Color Histogram RGB
- Color Histogram HSV
- GLCM texture features
- LBP texture features

## 1. Setup Environment

Jalankan notebook ini dari folder project `tomato-leaf-svm-knn`. Jika dependency belum terpasang, jalankan dulu di terminal:

```bash
python3.12 -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements.txt
```

In [ ]:
from pathlib import Path
import sys

# Jika notebook dibuka dari folder notebooks, pindahkan konteks ke project root.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

In [ ]:
import pandas as pd
from IPython.display import Markdown, display, Image

from src import config
from src.analyze_dataset import analyze_dataset
from src.prepare_dataset import prepare_dataset
from src.feature_extraction import extract_features
from src.train_svm import train_svm
from src.train_knn import train_knn
from src.evaluate_models import evaluate_models
from src.compare_results import compare_results

print("Dataset directory:", config.DATASET_DIR)
print("Target classes:", config.TARGET_CLASSES)
print("Image size:", config.IMAGE_SIZE)
print("Random state:", config.RANDOM_STATE)

## 2. Analisis Jurnal

Ringkasan jurnal sudah dibuat di `reports/analisis_jurnal.md`. Cell berikut menampilkannya di notebook.

In [ ]:
display(Markdown((PROJECT_ROOT / "reports/analisis_jurnal.md").read_text(encoding="utf-8")))

## 3. Analisis Dataset Lokal

Tahap ini membaca struktur folder dataset, menghitung jumlah gambar per kelas, memeriksa format file, memeriksa gambar rusak, dan menghitung statistik ukuran gambar.

Output:

- `reports/dataset_summary.csv`
- `reports/dataset_analysis.md`
- `reports/dataset_files_detail.csv`
- `reports/corrupt_images.csv`

In [ ]:
dataset_summary = analyze_dataset()
dataset_summary

In [ ]:
display(Markdown((PROJECT_ROOT / "reports/dataset_analysis.md").read_text(encoding="utf-8")))

## 4. Preprocessing dan Split Dataset

Tahap ini:

1. Memfilter lima kelas target jurnal.
2. Mengambil maksimal 1000 gambar per kelas.
3. Resize gambar ke ukuran seragam.
4. Membuat split stratified 80/20.

Output:

- `data/processed/dataset_metadata.csv`
- `data/processed/train_metadata.csv`
- `data/processed/test_metadata.csv`
- `data/processed/images/`

In [ ]:
metadata = prepare_dataset()
metadata.head()

In [ ]:
metadata.groupby(["split", "label"]).size().to_frame("jumlah_gambar")

## 5. Ekstraksi Fitur Citra

SVM dan KNN membutuhkan vektor numerik, sehingga gambar tidak langsung dimasukkan sebagai citra mentah. Project ini mengubah setiap gambar menjadi fitur eksplisit.

Output:

- `data/features/features.csv`
- `data/features/labels.csv`
- `data/features/feature_names.csv`

In [ ]:
features = extract_features()
features.shape

In [ ]:
feature_names = pd.read_csv(PROJECT_ROOT / "data/features/feature_names.csv")
feature_names.head(20)

## 6. Training dan Cross Validation SVM

SVM menggunakan pipeline:

- `StandardScaler`
- `SVC(kernel="rbf", probability=True)`

Evaluasi cross validation dilakukan pada 5-Fold, 10-Fold, dan 20-Fold.

Output:

- `models/svm_model.pkl`
- `reports/svm_results.csv`

In [ ]:
svm_results = train_svm()
svm_results

## 7. Training dan Cross Validation KNN

KNN menggunakan pipeline:

- `StandardScaler`
- `KNeighborsClassifier(metric="euclidean")`

Eksperimen dilakukan untuk `k=3,5,7,9`. Model final disimpan menggunakan default `k=5`.

Output:

- `models/knn_model.pkl`
- `reports/knn_results.csv`

In [ ]:
knn_results = train_knn()
knn_results

In [ ]:
# Melihat eksperimen KNN terbaik dari semua variasi k.
knn_results.sort_values("accuracy_mean", ascending=False).head(10)

## 8. Evaluasi Test Set

Tahap ini mengevaluasi model final pada test set 20%.

Metrik:

- Accuracy
- Precision macro
- Recall macro
- F1 macro
- AUC multiclass OVR macro
- Classification report
- Confusion matrix

Output:

- `reports/evaluation_svm.txt`
- `reports/evaluation_knn.txt`
- `reports/test_evaluation_summary.csv`
- `reports/figures/confusion_matrix_svm.png`
- `reports/figures/confusion_matrix_knn.png`

In [ ]:
test_evaluation = evaluate_models()
test_evaluation

In [ ]:
print((PROJECT_ROOT / "reports/evaluation_svm.txt").read_text(encoding="utf-8"))

In [ ]:
print((PROJECT_ROOT / "reports/evaluation_knn.txt").read_text(encoding="utf-8"))

### Confusion Matrix SVM

In [ ]:
Image(filename=str(PROJECT_ROOT / "reports/figures/confusion_matrix_svm.png"))

### Confusion Matrix KNN

In [ ]:
Image(filename=str(PROJECT_ROOT / "reports/figures/confusion_matrix_knn.png"))

## 9. Perbandingan Hasil SVM vs KNN

Tahap ini membuat tabel perbandingan seperti jurnal dan menentukan model terbaik berdasarkan hasil cross validation.

Output:

- `reports/results_summary.md`
- `reports/results_comparison.csv`
- `reports/figures/model_comparison_accuracy.png`
- `reports/figures/model_comparison_precision.png`
- `reports/figures/model_comparison_recall.png`

In [ ]:
comparison = compare_results()
comparison

In [ ]:
display(Markdown((PROJECT_ROOT / "reports/results_summary.md").read_text(encoding="utf-8")))

### Grafik Perbandingan Accuracy

In [ ]:
Image(filename=str(PROJECT_ROOT / "reports/figures/model_comparison_accuracy.png"))

### Grafik Perbandingan Precision

In [ ]:
Image(filename=str(PROJECT_ROOT / "reports/figures/model_comparison_precision.png"))

### Grafik Perbandingan Recall

In [ ]:
Image(filename=str(PROJECT_ROOT / "reports/figures/model_comparison_recall.png"))

## 10. Kesimpulan Notebook

Notebook ini mengikuti alur replikasi jurnal:

1. Analisis jurnal.
2. Analisis dataset lokal.
3. Preprocessing dan split 80/20.
4. Ekstraksi fitur citra eksplisit.
5. Training SVM dan KNN.
6. Cross validation 5/10/20-Fold.
7. Evaluasi test set.
8. Perbandingan hasil dan kesimpulan model terbaik.

Catatan: hasil bisa berbeda dari jurnal karena jurnal tidak menjelaskan detail preprocessing dan ekstraksi fitur sebelum SVM/KNN.